# Accuracy

This notebook computes accuracy for the current saved checkpoint using the processed test split.


In [1]:
import sys
from pathlib import Path

import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.inference.resume_screener import ResumeScreeningModel
from ml.training.train_model import DEFAULT_THRESHOLD, load_processed_tensors

processed_dir = PROJECT_ROOT / "data" / "processed"
model_path = PROJECT_ROOT / "ml" / "models" / "resume_net.pt"

_, X_test, _, y_test = load_processed_tensors(processed_dir=processed_dir)
model = ResumeScreeningModel(str(model_path), device="cpu", threshold=DEFAULT_THRESHOLD)

with torch.no_grad():
    probabilities = torch.sigmoid(model.model(X_test)).detach().cpu().numpy().reshape(-1)

labels = y_test.cpu().numpy().reshape(-1).astype(int)
predictions = (probabilities >= DEFAULT_THRESHOLD).astype(int)

print(f"Threshold: {DEFAULT_THRESHOLD}")
print(f"Test size: {labels.shape[0]}")


Threshold: 0.3
Test size: 6000


In [2]:
accuracy = round(float(accuracy_score(labels, predictions)), 4)
print(f"Accuracy: {accuracy:.4f}")


Accuracy: 0.6988
